# Notebook 2 — Semantic Exploration (CountVectorizer + Sentence Embeddings + UMAP)

> Goal: prepare reproducible data for semantic analysis, extract interpretable vocabulary by class, and visualize 2D structure with UMAP.

## 1) Environment setup and dependencies
Install/import libraries, set random seed, define paths, and configure visualization/logging options.

In [4]:
import logging
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import normalize
import umap.umap_ as umap
from sentence_transformers import SentenceTransformer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

logging.basicConfig(level=logging.INFO, format="%(levelname)s - %(message)s")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 180)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from political_bias_api.core.paths import PROCESSED_DIR, USERS_CSV

ARTIFACTS_DIR = PROCESSED_DIR / "semantic_artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Artifacts dir: {ARTIFACTS_DIR}")

Artifacts dir: /Users/sandrasanchezp/projects/nlp-political-bias-api/data/processed/semantic_artifacts


## 2) Data loading and schema validation (user-level only)
Read and validate only the user-level dataset in this phase. Post-level data will be added in a later section/iteration.

In [5]:
required_user_cols = {"user_id", "label", "n_posts", "text"}

users_df = pd.read_csv(USERS_CSV)
print(f"Users dataset: {USERS_CSV}")
print(f"Shape: {users_df.shape}")

missing_user_cols = required_user_cols - set(users_df.columns)
assert not missing_user_cols, f"Missing user-level columns: {missing_user_cols}"

display(users_df.dtypes.to_frame("dtype"))
display((users_df.isna().mean() * 100).sort_values(ascending=False).to_frame("missing_%"))
print("Duplicated user_id:", int(users_df["user_id"].duplicated().sum()))
print("Post-level loading deferred to a later notebook section.")

Users dataset: /Users/sandrasanchezp/projects/nlp-political-bias-api/data/processed/reddit_users_text_label.csv
Shape: (4150, 4)


,dtype
user_id,object
label,int64
n_posts,int64
text,object


,missing_%
user_id,0.0
label,0.0
n_posts,0.0
text,0.0


Duplicated user_id: 0
Post-level loading deferred to a later notebook section.


## 3) Analysis dataset assembly (no additional preprocessing)
Use the processed user-level dataset as-is and derive analysis-only features for exploration.

In [ ]:
MAX_USERS = 4000

work_df = users_df[["user_id", "label", "n_posts", "text"]].copy()
work_df["granularity"] = "user"

text_series = work_df["text"].fillna("").astype(str)
work_df["text_len_chars"] = text_series.str.len()
work_df["text_len_words"] = text_series.str.split().str.len()

work_sample_df = work_df
if len(work_df) > MAX_USERS:
    n_classes = work_df["label"].nunique()
    per_class = max(1, MAX_USERS // n_classes)
    sampled_parts = []
    for _, group in work_df.groupby("label", group_keys=False):
        sampled_parts.append(group.sample(min(len(group), per_class), random_state=SEED))
    work_sample_df = pd.concat(sampled_parts, ignore_index=True)

print("Using processed user-level data as-is (no extra cleaning).")
print(f"Full working shape: {work_df.shape}")
print(f"Sample shape for analysis: {work_sample_df.shape}")
print("Granularity:", work_df["granularity"].iloc[0])
display(work_sample_df.head(3))

/var/folders/69/_plt7pmj5hj1scqbf0lqwmsm0000gn/T/ipykernel_4351/4061576510.py:15: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  work_df.groupby("label", group_keys=False)


Using processed user-level data as-is (no extra cleaning).
Full working shape: (4150, 7)
Sample shape for analysis: (3086, 7)
Granularity: user


,user_id,label,n_posts,text,granularity,text_len_chars,text_len_words
0,61e3ab21ea343fbc9fa62a7d6dc30b8da95012c9902d67c0660f576b7d591ddb,0,1063,"That's what I wish a lot of people realized. When the Democratic senate refused to, say, confirm Bork for Reagan, it wasn't a debate if Reagan got to fill the seat. It was a ...",user,337475,58718
1,c79ffcab2b800ba115d9d9303cfd7baa63bc4e769f4e11bd38478ce66342e7b0,0,1548,"Yeah, they don't care about people. They care about clutching their pearls.\nJoe Biden Leads President Donald Trump In Pennsylvania In 2020 Election, Poll Shows\nBiden's origin...",user,181656,29905
2,abf1b46a29cf0999422b71a380fc5c36773ac299a35de9293b439050bd7e9231,0,316,It's almost like the entire Trump admin is an organized crime enterprise.\nAnyone find it absolutely bizarre that Trump supporters are so heavily invested in HCQ? It's like th...,user,106680,17242
